# ✅ Enjazi | إنجــازي
**Write naturally. Get organized. Know what matters.** · مساعدك الذكي لتنظيم مهامك وإنجاز يومك

Prototype that runs in Google Colab with **the model: Qwen2.5**, and a simple Gradio app.

| Page | Function |
|---|---|
| ✍️ Add tasks | Write a free-form note in Arabic or English; Qwen turns it into clear tasks |
| 🎯 Priorities | Shows what to do first and why; you can override any priority |
| ✅ Check | Tick tasks as done, move them to tomorrow, or delete them |
| 💬 Assistant | Chat with Qwen: "what should I focus on?", "I have 2 free hours", "move X to tomorrow" |
| 📊 Dashboard | Progress, workload for the next 7 days, tasks by category and by priority |

The interface language (English / العربية) is chosen with the switch under the banner. One language at a time.

In [1]:
# Cell 1 — install
!pip -q install "gradio>=5,<6" plotly pandas transformers accelerate

## 1 · Load the model (Qwen2.5)

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
print("Loaded:", MODEL_ID, "on", model.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct on cuda:0


## 2 · Settings and text (English / العربية)

In [3]:
import re, json, html
from datetime import date, datetime, timedelta, timezone
import plotly.graph_objects as go
import gradio as gr

TZ_HOURS = 3   # Riyadh = UTC+3.

CATS = {"study": ("📚", "#8b5cf6"), "work": ("💼", "#3b82f6"), "health": ("💪", "#10b981"),
        "errands": ("🛒", "#f59e0b"), "social": ("💬", "#ec4899"), "personal": ("🌿", "#14b8a6")}
PRIO_COLORS = {1: "#f43f5e", 2: "#fb923c", 3: "#38bdf8", 4: "#34d399"}

# Every on-screen text, one language at a time: key -> (English, Arabic).
_TEXT = {
    "title": ("Write naturally. Get organized. Know what matters.", "مساعدك الذكي لتنظيم مهامك وإنجاز يومك"),
    "tab_add": ("✍️ Add tasks", "✍️ تسجيل المهام"), "tab_prio": ("🎯 Priorities", "🎯 الأولويات"),
    "tab_check": ("✅ Check", "✅ إنجاز"), "tab_chat": ("💬 Assistant", "💬 المساعد"), "tab_dash": ("📊 Dashboard", "📊 لوحة التقدم"),
    "note": ("Write your tasks the way you talk", "اكتب مهامك كما تتكلم"),
    "note_ph": ("e.g. I have an exam next week and must finish the project tomorrow", "مثال: عندي اختبار الأسبوع الجاي ولازم أخلص المشروع بكرة"),
    "add": ("✨ Organize my tasks", "✨ رتّب مهامي"), "try": ("Try:", "جرّب:"),
    "found": ("Added {n} task(s)", "تمت إضافة {n} مهمة"), "empty_found": ("No task found in that text.", "لم أجد مهمة في هذا النص."),
    "write_first": ("Write something first.", "اكتب شيئًا أولاً."),
    "model_fail": ("Qwen could not read the answer; the note was saved as one task.", "تعذّر على Qwen فهم الرد، فحُفظت الملاحظة كمهمة واحدة."),
    "first": ("Do this first", "ابدأ بهذه أولاً"), "clear": ("All clear! Add tasks in the first tab.", "لا توجد مهام! أضف مهامك من الصفحة الأولى."),
    "pick": ("Choose a task", "اختر مهمة"), "prio": ("Priority", "الأولوية"), "set_prio": ("Set priority", "تعيين الأولوية"),
    "auto": ("Automatic", "تلقائي"), "p1": ("Critical", "حرجة"), "p2": ("High", "عالية"), "p3": ("Medium", "متوسطة"), "p4": ("Low", "منخفضة"),
    "check": ("Tick the tasks you want to update", "حدّد المهام التي تريد تحديثها"),
    "done": ("✅ Done", "✅ تم"), "tmr": ("⏭ Move to tomorrow", "⏭ إلى الغد"), "del": ("🗑 Delete", "🗑 حذف"),
    "pick_first": ("Select at least one task.", "حدّد مهمة واحدة على الأقل."), "well_done": ("🎉 Well done!", "🎉 أحسنت!"),
    "finished": ("Finished", "المنجزة"), "nothing_done": ("Nothing finished yet.", "لم تُنجز شيئًا بعد."),
    "chat": ("Talk to Enjazi", "تحدث مع إنجازي"), "chat_ph": ("What should I focus on today?", "وش أبدأ فيه اليوم؟"),
    "send": ("Send", "إرسال"),
    "kpi_total": ("Total", "الكل"), "kpi_done": ("Done", "منجزة"), "kpi_open": ("Open", "مفتوحة"), "kpi_rate": ("Completion", "الإنجاز"),
    "c_load": ("Workload · next 7 days", "الحمل · الأيام السبعة القادمة"), "c_cat": ("By category", "حسب الفئة"),
    "c_prio": ("Open tasks by priority", "المهام المفتوحة حسب الأولوية"), "no_data": ("Nothing to show yet", "لا توجد بيانات بعد"),
    "cheer0": ("A fresh start. Your first win is one task away. 🌱", "بداية جديدة. إنجازك الأول على بُعد مهمة. 🌱"),
    "cheer1": ("{n} task(s) finished. Keep going! 🎉", "أنجزت {n} مهمة. استمر! 🎉"),
    "cheer2": ("Everything is finished. Amazing work! 🏆", "أنجزت كل شيء. عمل رائع! 🏆"),
    "today": ("Today", "اليوم"), "tomorrow": ("Tomorrow", "غدًا"), "nodate": ("No date", "بدون موعد"), "past": ("Past date", "فات موعدها"),
    "r_past": ("its date has passed, so it is worth rescheduling", "فات موعدها فالأفضل إعادة جدولتها"),
    "r_today": ("it is due today", "موعدها اليوم"), "r_tmr": ("it is due tomorrow", "موعدها غدًا"),
    "r_days": ("it is due in {d} days", "موعدها بعد {d} أيام"), "r_imp5": ("it matters a lot", "مهمة جدًا"), "r_imp4": ("it matters", "مهمة"),
    "r_manual": ("you chose this priority", "أنت حددت هذه الأولوية"), "r_none": ("it has no deadline, so it can wait", "لا موعد نهائي لها فيمكنها الانتظار"),
    "because": ("Because ", "لأن "), "and": (" and ", " و"),
    "a_start": ("Start with “{t}”. {r}", "ابدأ بـ«{t}». {r}"), "a_next": (" Next: {x}.", " بعدها: {x}."),
    "a_none": ("You have no open tasks. 🎉", "لا توجد مهام مفتوحة. 🎉"),
    "a_free": ("With {m} free minutes, these fit: {x}.", "في {m} دقيقة متاحة تناسبك: {x}."),
    "a_nofit": ("Nothing fits in that time. Enjoy the break! ☕", "لا شيء يناسب هذا الوقت. استمتع بالاستراحة! ☕"),
    "a_done": ("Marked “{t}” as done. ✅", "تم إنجاز «{t}». ✅"), "a_del": ("Deleted “{t}”.", "تم حذف «{t}»."),
    "a_move": ("Moved “{t}” to {d}.", "نُقلت «{t}» إلى {d}."), "a_add": ("Added: {x}.", "تمت إضافة: {x}."),
    "a_which": ("Which task do you mean?", "أي مهمة تقصد؟"), "a_other": ("I can help with priorities, free time, and updating tasks.", "أقدر أساعدك في الأولويات ووقت الفراغ وتحديث المهام."),
}
TEXT = {"en": {k: v[0] for k, v in _TEXT.items()}, "ar": {k: v[1] for k, v in _TEXT.items()}}
EXAMPLES = {
    "en": ["I have an exam next week and I must finish the project tomorrow",
           "Remind me to submit the report tomorrow at 3pm",
           "buy groceries today, call the dentist on Thursday, maybe read a book"],
    "ar": ["عندي اختبار الأسبوع الجاي ولازم أخلص المشروع بكرة",
           "ذكرني أسلّم التقرير بكرة الساعة 3 العصر",
           "أبغى أشتري بقالة اليوم، وأكلم الدكتور يوم الخميس، وممكن أقرأ كتاب"]}
CAT_NAMES = {"en": {"study": "Study", "work": "Work", "health": "Health", "errands": "Errands", "social": "Social", "personal": "Personal"},
             "ar": {"study": "دراسة", "work": "عمل", "health": "صحة", "errands": "مشاوير", "social": "اجتماعي", "personal": "شخصي"}}
WEEKDAYS = {"en": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], "ar": ["الاثنين", "الثلاثاء", "الأربعاء", "الخميس", "الجمعة", "السبت", "الأحد"]}

## 3 · Qwen: understanding notes and chat commands
Qwen answers in strict JSON. For dates it only returns **"days from today"**; Python computes the real date (small models are weak at calendar math).
If Qwen's answer is unreadable, the app never breaks: the note is saved as one plain task.

In [4]:
def qwen(system, user, max_new_tokens=450):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def ask_json(system, user):
    raw = qwen(system, user)
    try:
        return json.loads(raw[raw.index("{"): raw.rindex("}") + 1])
    except Exception:
        return None

def now_local():
    return datetime.now(timezone.utc).replace(tzinfo=None) + timedelta(hours=TZ_HOURS)

def calendar_hint(today):
    names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    days = ", ".join(f"{n}={(i - today.weekday()) % 7 or 7}" for i, n in enumerate(names))
    return f"Today is {today} ({names[today.weekday()]}). Days until the next weekday: {days}."

EXTRACT_PROMPT = """You are the task-understanding engine of Enjazi, a bilingual (Arabic/English) to-do app.
{CAL}
Split the user's note (Arabic, English or mixed, casual dialect is common) into separate tasks.
Reply with JSON only:
{"tasks":[{"title":str,"due_in_days":int|null,"time":"HH:MM"|null,"importance":1-5,"minutes":int,"category":"study|work|health|errands|social|personal"}]}
Rules:
- title: short, in the user's own language and words, without date or time words. Never translate.
- due_in_days: 0 today, 1 tomorrow (بكرة), 2 day after tomorrow (بعد بكرة), 7 next week (الأسبوع الجاي). Use the table above for weekday names. null if no date.
- time: 24-hour. "at 3" or "الساعة 3" without am/pm means 15:00. "after work"/"بعد الدوام" means 17:00. Otherwise null.
- importance: 5 exams, deadlines, urgent. 4 reports, meetings, doctor. 3 normal. 2 optional ("maybe", "ممكن").
- minutes: a realistic estimate.
Examples:
Note: عندي اختبار الأسبوع الجاي ولازم أخلص المشروع بكرة
{"tasks":[{"title":"اختبار","due_in_days":7,"time":null,"importance":5,"minutes":90,"category":"study"},{"title":"أخلص المشروع","due_in_days":1,"time":null,"importance":4,"minutes":120,"category":"work"}]}
Note: Remind me to submit the report tomorrow at 3
{"tasks":[{"title":"Submit the report","due_in_days":1,"time":"15:00","importance":4,"minutes":60,"category":"work"}]}"""

ASSIST_PROMPT = """You are the assistant of Enjazi, a bilingual (Arabic/English) to-do app.
{CAL}
The user's open tasks (id | title | due | minutes | priority):
{TASKS}
Decide what the user wants. Reply with JSON only:
{"action":"focus|free_time|done|delete|move|add|other","task_id":int|null,"minutes":int|null,"due_in_days":int|null}
- focus: asks what to do first or what to focus on.
- free_time: says how much free time they have; minutes = that time in minutes.
- done, delete, move: refers to one of the tasks above; task_id is its id. For move, due_in_days is the new date (default 1).
- add: wants to add a new task ("remind me to ...", "ذكرني ...", "أضف ...").
- other: anything else.
Examples:
What should I focus on today? -> {"action":"focus","task_id":null,"minutes":null,"due_in_days":null}
I have two free hours -> {"action":"free_time","task_id":null,"minutes":120,"due_in_days":null}
عندي نص ساعة فاضية -> {"action":"free_time","task_id":null,"minutes":30,"due_in_days":null}
I finished the report (report has id 2) -> {"action":"done","task_id":2,"minutes":null,"due_in_days":null}
انقل المشروع لبعد بكرة (المشروع id 1) -> {"action":"move","task_id":1,"minutes":null,"due_in_days":2}"""

def clean_task(p, today):
    title = str(p.get("title") or "").strip()
    if not title:
        return None
    due = None
    try:
        if p.get("due_in_days") is not None:
            due = (today + timedelta(days=max(0, min(365, int(p["due_in_days"]))))).isoformat()
    except Exception:
        pass
    tm = p.get("time")
    m = re.match(r"^(\d{1,2}):(\d{2})$", tm) if isinstance(tm, str) else None
    tm = f"{int(m.group(1)):02d}:{m.group(2)}" if m and int(m.group(1)) < 24 else None
    num = lambda v, lo, hi, d: max(lo, min(hi, int(v))) if str(v).lstrip("-").isdigit() else d
    return {"title": title, "due": due, "time": tm, "importance": num(p.get("importance"), 1, 5, 3),
            "est": num(p.get("minutes"), 5, 480, 30), "category": p.get("category") if p.get("category") in CATS else "personal"}

def extract(text, T):
    today = now_local().date()
    data = ask_json(EXTRACT_PROMPT.replace("{CAL}", calendar_hint(today)), "Note: " + text)
    tasks = [t for t in (clean_task(p, today) for p in (data or {}).get("tasks", []) if isinstance(p, dict)) if t]
    if not tasks:
        gr.Warning(T["model_fail"])
        tasks = [clean_task({"title": text.strip()}, today)]
    return tasks

## 4 · Tasks, priorities and reasons
The state is a plain Python dict inside the session (no database).

In [5]:
def new_state():
    return {"seq": 0, "tasks": [], "lang": "en"}

def add_tasks(st, parsed, original):
    out = []
    for p in parsed:
        st["seq"] += 1
        t = {**p, "id": st["seq"], "original": original, "status": "todo", "manual": None, "done_at": None}
        st["tasks"].append(t); out.append(t)
    return out

def open_tasks(st):
    return [t for t in st["tasks"] if t["status"] == "todo"]

def days_to(t, today):
    return (date.fromisoformat(t["due"]) - today).days if t["due"] else None

def score(t, today):
    d = days_to(t, today)
    urgency = 15 if d is None else 100 if d < 0 else 90 if d == 0 else 70 if d == 1 else 50 if d <= 3 else 30 if d <= 7 else 12
    return urgency + t["importance"] * 8

def level(t, today):
    if t["manual"]: return t["manual"]
    s = score(t, today)
    return 1 if s >= 120 else 2 if s >= 90 else 3 if s >= 55 else 4

def ranked(st):
    today = now_local().date()
    return sorted(open_tasks(st), key=lambda t: (level(t, today), -score(t, today)))

def reason(t, T):
    today, parts = now_local().date(), []
    d = days_to(t, today)
    if d is not None:
        parts.append(T["r_past"] if d < 0 else T["r_today"] if d == 0 else T["r_tmr"] if d == 1 else T["r_days"].format(d=d) if d <= 7 else None)
    if t["importance"] >= 5: parts.append(T["r_imp5"])
    elif t["importance"] == 4: parts.append(T["r_imp4"])
    if t["manual"]: parts.append(T["r_manual"])
    parts = [p for p in parts if p] or [T["r_none"]]
    return T["because"] + T["and"].join(parts) + "."

def due_text(t, T):
    d = days_to(t, now_local().date())
    base = T["nodate"] if d is None else T["past"] if d < 0 else T["today"] if d == 0 else T["tomorrow"] if d == 1 else t["due"]
    return base + (f" · {t['time']}" if t["time"] else "")

def complete(t):
    t["status"], t["done_at"] = "done", now_local().isoformat()

## 5 · Smart Task Assistant

In [6]:
def assistant(st, msg):
    T, today = TEXT[st["lang"]], now_local().date()
    pool = ranked(st)
    listing = "\n".join(f"{t['id']} | {t['title']} | {t['due'] or '-'} | {t['est']} | P{level(t, today)}" for t in pool) or "(none)"
    data = ask_json(ASSIST_PROMPT.replace("{CAL}", calendar_hint(today)).replace("{TASKS}", listing), msg) or {}
    action = data.get("action", "other")
    task = next((t for t in pool if str(t["id"]) == str(data.get("task_id"))), None)

    if action == "focus":
        if not pool: return T["a_none"]
        more = ", ".join(t["title"] for t in pool[1:3])
        return T["a_start"].format(t=pool[0]["title"], r=reason(pool[0], T)) + (T["a_next"].format(x=more) if more else "")
    if action == "free_time":
        mins, fit, used = int(data.get("minutes") or 60), [], 0
        for t in pool:
            if used + t["est"] <= mins: fit.append(f"{t['title']} ({t['est']}m)"); used += t["est"]
        return T["a_free"].format(m=mins, x=", ".join(fit)) if fit else T["a_nofit"]
    if action in ("done", "delete", "move"):
        if not task: return T["a_which"]
        if action == "done": complete(task); return T["a_done"].format(t=task["title"])
        if action == "delete": st["tasks"].remove(task); return T["a_del"].format(t=task["title"])
        d = today + timedelta(days=int(data.get("due_in_days") or 1))
        task["due"] = d.isoformat()
        return T["a_move"].format(t=task["title"], d=d.strftime("%a %d %b"))
    if action == "add":
        created = add_tasks(st, extract(msg, T), msg)
        return T["a_add"].format(x=", ".join(t["title"] for t in created))
    return T["a_other"]

## 6 · Cards and dashboard

In [7]:
def esc(x): return html.escape(str(x))

def wrap(inner, lang):
    return f"<div dir='{'rtl' if lang == 'ar' else 'ltr'}'>{inner}</div>"

def task_card(t, st, big=False):
    lang, today = st["lang"], now_local().date()
    T, lv = TEXT[lang], level(t, now_local().date())
    emo, ccol = CATS[t["category"]]
    pcol = PRIO_COLORS[lv]
    chip = lambda txt, c: f"<span class='chip' style='--c:{c}'>{txt}</span>"
    return (f"<div class='task {'big' if big else ''}' style='--p:{pcol}'>"
            f"<div class='row'><div class='ttl'>{esc(t['title'])}</div>{chip(T['p%d' % lv], pcol)}</div>"
            f"<div class='meta'>{chip(emo + ' ' + CAT_NAMES[lang][t['category']], ccol)}{chip('📅 ' + due_text(t, T), '#6366f1')}{chip('⏱ ' + str(t['est']) + 'm', '#0ea5e9')}</div>"
            f"<div class='why'>💡 {reason(t, T)}</div></div>")

def render_focus(st):
    T, pool = TEXT[st["lang"]], ranked(st)
    if not pool:
        return wrap(f"<div class='hero ok'><h3>{T['clear']}</h3></div>", st["lang"])
    return wrap(f"<div class='hero'><div class='eyebrow'>🎯 {T['first']}</div>{task_card(pool[0], st, True)}</div>", st["lang"])

def render_board(st):
    pool = ranked(st)
    return wrap("<div class='grid'>" + "".join(task_card(t, st) for t in pool[1:]) + "</div>", st["lang"]) if len(pool) > 1 else ""

def render_new(created, st):
    T = TEXT[st["lang"]]
    if not created: return wrap(f"<div class='empty'>{T['empty_found']}</div>", st["lang"])
    return wrap(f"<div class='eyebrow'>✨ {T['found'].format(n=len(created))}</div><div class='grid'>"
                + "".join(task_card(t, st) for t in created) + "</div>", st["lang"])

def render_done(st):
    T = TEXT[st["lang"]]
    done = [t for t in st["tasks"] if t["status"] == "done"][-8:]
    body = "".join(f"<div class='dn'>✅ {esc(t['title'])}</div>" for t in reversed(done)) or f"<div class='empty'>{T['nothing_done']}</div>"
    return wrap(f"<div class='eyebrow'>{T['finished']}</div>{body}", st["lang"])

def render_kpis(st):
    T, total = TEXT[st["lang"]], len(st["tasks"])
    done = total - len(open_tasks(st))
    rate = round(100 * done / total) if total else 0
    items = [(total, T["kpi_total"], "#6366f1"), (done, T["kpi_done"], "#10b981"), (len(open_tasks(st)), T["kpi_open"], "#0ea5e9"), (f"{rate}%", T["kpi_rate"], "#ec4899")]
    cards = "".join(f"<div class='kpi' style='--c:{c}'><div class='num'>{n}</div><div>{l}</div></div>" for n, l, c in items)
    msg = T["cheer0"] if done == 0 else T["cheer2"] if done == total else T["cheer1"].format(n=done)
    bar = f"<div class='bar'><div style='width:{rate}%'></div></div>"
    return wrap(f"<div class='kpis'>{cards}</div>{bar}<div class='cheer'>{msg}</div>", st["lang"])

def _chart(fig, title, empty_msg=None):
    if empty_msg:
        fig.add_annotation(text=empty_msg, showarrow=False, font=dict(size=14, color="#94a3b8"))
        fig.update_layout(xaxis_visible=False, yaxis_visible=False)
    fig.update_layout(title=title, height=300, margin=dict(l=10, r=10, t=50, b=10), paper_bgcolor="rgba(0,0,0,0)",
                      plot_bgcolor="rgba(0,0,0,0)", font=dict(color="#64748b"), showlegend=False)
    return fig

def fig_load(st):
    T, lang, today = TEXT[st["lang"]], st["lang"], now_local().date()
    days = [today + timedelta(i) for i in range(7)]
    counts = [sum(1 for t in open_tasks(st) if t["due"] and date.fromisoformat(t["due"]) == d) for d in days]
    fig = go.Figure(go.Bar(x=[WEEKDAYS[lang][d.weekday()] for d in days], y=counts, text=counts, textposition="outside",
                           marker=dict(color=counts, colorscale=[[0, "#c4b5fd"], [1, "#ec4899"]])))
    fig.update_yaxes(dtick=1, gridcolor="rgba(148,163,184,.25)")
    return _chart(fig, T["c_load"], None if any(counts) else T["no_data"])

def fig_cat(st):
    T, lang = TEXT[st["lang"]], st["lang"]
    cnt = {}
    for t in st["tasks"]: cnt[t["category"]] = cnt.get(t["category"], 0) + 1
    fig = go.Figure(go.Pie(labels=[f"{CATS[k][0]} {CAT_NAMES[lang][k]}" for k in cnt], values=list(cnt.values()), hole=.55,
                           marker=dict(colors=[CATS[k][1] for k in cnt]), textinfo="value")) if cnt else go.Figure()
    return _chart(fig, T["c_cat"], None if cnt else T["no_data"])

def fig_prio(st):
    T, today = TEXT[st["lang"]], now_local().date()
    counts = [sum(1 for t in open_tasks(st) if level(t, today) == p) for p in (1, 2, 3, 4)]
    fig = go.Figure(go.Bar(x=[T[f"p{p}"] for p in (1, 2, 3, 4)], y=counts, text=counts, textposition="outside", marker=dict(color=list(PRIO_COLORS.values()))))
    fig.update_yaxes(dtick=1, gridcolor="rgba(148,163,184,.25)")
    return _chart(fig, T["c_prio"], None if any(counts) else T["no_data"])

def render_all(st):
    """Everything that changes when tasks change (the language labels are included)."""
    T = TEXT[st["lang"]]
    choices = [(f"{t['title']}  ·  {due_text(t, T)}", t["id"]) for t in ranked(st)]
    return (render_focus(st), render_board(st), gr.Dropdown(choices=choices, value=None, label=T["pick"]),
            gr.CheckboxGroup(choices=choices, value=[], label=T["check"]),
            render_done(st), render_kpis(st), fig_load(st), fig_cat(st), fig_prio(st))

## 7 · The app
Five clear pages. With language switch between the languages Arabic & English enabled for user's preference.

In [8]:
CSS = """
.gradio-container{max-width:1050px!important;margin:auto}
.banner{border-radius:20px;padding:22px 28px;background:linear-gradient(120deg,#7c3aed,#ec4899 60%,#f59e0b)}
.banner h1,.banner p,.banner span{color:#fff!important}
.banner h1{margin:0;font-size:32px}
.banner p{margin:6px 0 0;font-size:17px}
.eyebrow{font-weight:700;margin:6px 0 10px}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:12px}
.task{border-radius:16px;padding:14px 16px;background:rgba(255,255,255,.85);color:#0f172a;border:1px solid rgba(148,163,184,.3);border-inline-start:6px solid var(--p);box-shadow:0 4px 16px rgba(15,23,42,.07)}
.dark .task{background:rgba(30,27,75,.6);color:#e2e8f0}
.task.big{padding:20px 22px}.task.big .ttl{font-size:24px}
.row{display:flex;justify-content:space-between;gap:10px;align-items:flex-start}
.ttl{font-weight:800;font-size:17px}.meta{display:flex;flex-wrap:wrap;gap:6px;margin:8px 0}.why{font-size:14px;opacity:.9}
.chip{padding:3px 10px;border-radius:999px;font-size:12.5px;font-weight:600;color:var(--c);background:color-mix(in srgb,var(--c) 14%,transparent);border:1px solid color-mix(in srgb,var(--c) 35%,transparent)}
.hero{border-radius:20px;padding:14px;background:linear-gradient(135deg,rgba(124,58,237,.12),rgba(236,72,153,.12))}
.hero.ok{text-align:center;padding:34px}.empty{padding:22px;text-align:center;opacity:.7;border:2px dashed rgba(148,163,184,.4);border-radius:16px}
.dn{padding:8px 14px;margin:5px 0;border-radius:12px;background:rgba(16,185,129,.12)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));gap:12px;margin-bottom:12px}
.kpi{border-radius:18px;padding:16px;text-align:center;color:#fff;background:linear-gradient(135deg,var(--c),color-mix(in srgb,var(--c) 55%,#ec4899))}
.kpi .num{font-size:34px;font-weight:900}
.bar{height:12px;border-radius:99px;background:rgba(148,163,184,.3);overflow:hidden}.bar div{height:100%;background:linear-gradient(90deg,#10b981,#6366f1)}
.cheer{padding:12px 16px;border-radius:14px;font-weight:600;background:rgba(99,102,241,.12);margin-top:10px}
"""
PRIO_KEYS = [(0, "auto"), (1, "p1"), (2, "p2"), (3, "p3"), (4, "p4")]

def banner(lang):
    return f"<div class='banner'><h1>✅ Enjazi | إنجــازي</h1><p>{TEXT[lang]['title']}</p></div>"

def build_ui():
    with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", secondary_hue="pink"), css=CSS, title="Enjazi | إنجازي") as demo:
        st = gr.State(new_state())
        head = gr.HTML(banner("en"))
        with gr.Row():
            gr.HTML("")
            lang = gr.Radio([("English", "en"), ("العربية", "ar")], value="en", label="🌐", container=False, scale=0, min_width=230)

        with gr.Tabs():
            with gr.Tab("✍️ Add tasks") as tab_add:
                note = gr.Textbox(lines=4, label=TEXT["en"]["note"], placeholder=TEXT["en"]["note_ph"])
                with gr.Row():
                    ex = [gr.Button(EXAMPLES["en"][i], size="sm") for i in range(3)]
                add_btn = gr.Button(TEXT["en"]["add"], variant="primary", size="lg")
                new_html = gr.HTML()
            with gr.Tab("🎯 Priorities") as tab_prio:
                focus_html = gr.HTML()
                board_html = gr.HTML()
                with gr.Row():
                    pick = gr.Dropdown(label=TEXT["en"]["pick"], choices=[], scale=3)
                    prio = gr.Dropdown(choices=[(TEXT["en"][k], v) for v, k in PRIO_KEYS], value=0, label=TEXT["en"]["prio"], scale=2)
                    prio_btn = gr.Button(TEXT["en"]["set_prio"], scale=1)
            with gr.Tab("✅ Check") as tab_check:
                check = gr.CheckboxGroup(label=TEXT["en"]["check"], choices=[])
                with gr.Row():
                    done_btn = gr.Button(TEXT["en"]["done"], variant="primary")
                    tmr_btn = gr.Button(TEXT["en"]["tmr"])
                    del_btn = gr.Button(TEXT["en"]["del"], variant="stop")
                done_html = gr.HTML()
            with gr.Tab("💬 Assistant") as tab_chat:
                chatbot = gr.Chatbot(type="messages", height=360, label=TEXT["en"]["chat"])
                msg = gr.Textbox(show_label=False, placeholder=TEXT["en"]["chat_ph"])
                send_btn = gr.Button(TEXT["en"]["send"], variant="primary")
            with gr.Tab("📊 Dashboard") as tab_dash:
                kpi_html = gr.HTML()
                with gr.Row():
                    p_load, p_cat, p_prio = gr.Plot(), gr.Plot(), gr.Plot()

        R = [focus_html, board_html, pick, check, done_html, kpi_html, p_load, p_cat, p_prio]
        LBL = [head, tab_add, tab_prio, tab_check, tab_chat, tab_dash, note, add_btn, *ex, prio, prio_btn,
               done_btn, tmr_btn, del_btn, chatbot, msg, send_btn]
        T_ = lambda s: TEXT[s["lang"]]

        def labels(l):
            T = TEXT[l]; rtl = l == "ar"
            return [banner(l), gr.Tab(label=T["tab_add"]), gr.Tab(label=T["tab_prio"]), gr.Tab(label=T["tab_check"]),
                    gr.Tab(label=T["tab_chat"]), gr.Tab(label=T["tab_dash"]),
                    gr.Textbox(label=T["note"], placeholder=T["note_ph"], rtl=rtl), gr.Button(T["add"]),
                    *[gr.Button(EXAMPLES[l][i]) for i in range(3)],
                    gr.Dropdown(choices=[(T[k], v) for v, k in PRIO_KEYS], value=0, label=T["prio"]), gr.Button(T["set_prio"]),
                    gr.Button(T["done"]), gr.Button(T["tmr"]), gr.Button(T["del"]),
                    gr.Chatbot(label=T["chat"], rtl=rtl), gr.Textbox(placeholder=T["chat_ph"], rtl=rtl), gr.Button(T["send"])]

        def on_lang(l, s):
            s["lang"] = l
            return [s, *labels(l), *render_all(s), ""]
        def on_add(text, s):
            if not (text or "").strip():
                gr.Warning(T_(s)["write_first"]); return [s, "", *render_all(s)]
            created = add_tasks(s, extract(text, T_(s)), text)
            return [s, render_new(created, s), *render_all(s)]
        def on_prio(pid, p, s):
            t = next((x for x in s["tasks"] if x["id"] == pid), None)
            if t: t["manual"] = int(p) or None
            return [s, *render_all(s)]
        def update_checked(action, ids, s):
            if not ids:
                gr.Warning(T_(s)["pick_first"]); return [s, *render_all(s)]
            for t in [x for x in s["tasks"] if x["id"] in ids]:
                if action == "done": complete(t)
                elif action == "tmr": t["due"] = (now_local().date() + timedelta(1)).isoformat()
                else: s["tasks"].remove(t)
            if action == "done": gr.Info(T_(s)["well_done"])
            return [s, *render_all(s)]
        def on_done(ids, s): return update_checked("done", ids, s)
        def on_tmr(ids, s): return update_checked("tmr", ids, s)
        def on_del(ids, s): return update_checked("del", ids, s)
        def on_chat(text, history, s):
            if not (text or "").strip(): return [s, history, "", *render_all(s)]
            reply = assistant(s, text)
            return [s, history + [{"role": "user", "content": text}, {"role": "assistant", "content": reply}], "", *render_all(s)]

        demo.load(on_lang, [lang, st], [st, *LBL, *R, new_html])
        lang.change(on_lang, [lang, st], [st, *LBL, *R, new_html])
        for i, b in enumerate(ex):
            b.click(lambda s, i=i: EXAMPLES[s["lang"]][i], [st], note)
        add_btn.click(on_add, [note, st], [st, new_html, *R])
        prio_btn.click(on_prio, [pick, prio, st], [st, *R])
        done_btn.click(on_done, [check, st], [st, *R]); tmr_btn.click(on_tmr, [check, st], [st, *R])
        del_btn.click(on_del, [check, st], [st, *R])
        send_btn.click(on_chat, [msg, chatbot, st], [st, chatbot, msg, *R]); msg.submit(on_chat, [msg, chatbot, st], [st, chatbot, msg, *R])
    return demo

## 8 · 🚀 Launch
Open the **public link** (`https://xxxx.gradio.live`) printed below.

In [9]:
demo = build_ui()
demo.queue().launch(share=True)

/tmp/ipykernel_9804/3124769330.py:30: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", secondary_hue="pink"), css=CSS, title="Enjazi | إنجازي") as demo:
/tmp/ipykernel_9804/3124769330.py:30: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", secondary_hue="pink"), css=CSS, title="Enjazi | إنجازي") as demo:
/tmp/ipykernel_9804/3124769330.py:59: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=360, label=TEXT["en"]["chat"])


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e1b5bb24afe6d6c38.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
